In [1]:
import pandas as pd
import numpy as np
import duckdb

In [2]:
conn = duckdb.connect("/Users/arjunprakashrao/Drive/projects/Willow-MC/data/willow.db")
# Query and convert
train_df = conn.execute("SELECT * FROM next_n_balls_features").df()

In [3]:
features = [
    "rr",
    "required_run_rate",
    "wickets_in_hand",
    "is_second_innings",
    "current_score",
    "overs_remaining"
]


In [4]:
train_df.columns

Index(['match_id', 'innings', 'is_second_innings', 'rr', 'required_run_rate',
       'overs_remaining', 'wickets_in_hand', 'current_score', 'target_total',
       'target_runs_next_6_balls', 'wickets_next_6_balls', 'wicket_event'],
      dtype='str')

In [5]:
import statsmodels.api as sm


X = train_df[features]
X = sm.add_constant(X)

y_runs = train_df["target_runs_next_6_balls"]

model_runs = sm.NegativeBinomial(y_runs, X).fit()
print(model_runs.summary())

Optimization terminated successfully.
         Current function value: 2.750421
         Iterations: 24
         Function evaluations: 34
         Gradient evaluations: 34
                        NegativeBinomial Regression Results                         
Dep. Variable:     target_runs_next_6_balls   No. Observations:               681444
Model:                     NegativeBinomial   Df Residuals:                   681437
Method:                                 MLE   Df Model:                            6
Date:                      Mon, 09 Mar 2026   Pseudo R-squ.:                 0.01535
Time:                              20:22:04   Log-Likelihood:            -1.8743e+06
converged:                             True   LL-Null:                   -1.9035e+06
Covariance Type:                  nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------

In [6]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.api import Logit
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

X = train_df[features]
y = train_df["wicket_event"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

wicket_model = Logit(y_train, X_train).fit()

y_pred_prob = wicket_model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print(wicket_model.summary())

Optimization terminated successfully.
         Current function value: 0.595294
         Iterations 5
Precision: 0.4169
Recall: 0.0031
                           Logit Regression Results                           
Dep. Variable:           wicket_event   No. Observations:               545155
Model:                          Logit   Df Residuals:                   545148
Method:                           MLE   Df Model:                            6
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                 0.01162
Time:                        20:22:19   Log-Likelihood:            -3.2453e+05
converged:                       True   LL-Null:                   -3.2834e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.4115      0.025    -16.595      0.000

In [7]:
import json

run_coeffs = model_runs.params.to_dict()

with open("run_model_coeffs.json", "w") as f:
    json.dump(run_coeffs, f)

wicket_coeffs = wicket_model.params.to_dict()

with open("wicket_model_coeffs.json", "w") as f:
    json.dump(wicket_coeffs, f)

In [ ]:
with open("/Users/arjunprakashrao/Drive/projects/Polymarket-lab/cricket_arbitrage/bots/run_model_coeffs.json", "w") as f:
    json.dump(run_coeffs, f)

with open("/Users/arjunprakashrao/Drive/projects/Polymarket-lab/cricket_arbitrage/bots/wicket_model_coeffs.json", "w") as f:
    json.dump(wicket_coeffs, f)

In [8]:
import json

with open("run_model_coeffs.json") as f:
    RUN = json.load(f)

with open("wicket_model_coeffs.json") as f:
    WICKET = json.load(f)

In [9]:
import numpy as np
from scipy.stats import nbinom
import math

def predict_mu(rr, rrr, overs_remaining, wih, is_second, score):

    z = (
        RUN["const"]
        + RUN["rr"] * rr
        + RUN["required_run_rate"] * rrr
        + RUN["overs_remaining"] * overs_remaining
        + RUN["wickets_in_hand"] * wih
        + RUN["is_second_innings"] * is_second
        + RUN["current_score"] * score
    )

    return np.exp(z)

def sample_runs(mu):

    alpha = RUN["alpha"]
    r = 1 / alpha
    p = r / (r + mu)

    return np.random.negative_binomial(r, p)

def predict_p_wicket(rr, rrr, overs_remaining, wih, is_second, score):

    z = (
        WICKET["const"]
        + WICKET["rr"] * rr
        + WICKET["required_run_rate"] * rrr
        + WICKET["overs_remaining"] * overs_remaining
        + WICKET["wickets_in_hand"] * wih
        + WICKET["is_second_innings"] * is_second
        + WICKET["current_score"] * score
    )

    return 1 / (1 + np.exp(-z))

# Monte Carlo

In [10]:
MAX_BALLS = 120
MAX_WICKETS = 10
STEP_BALLS = 6

In [11]:
initial = init_state(
    score=60,
    wickets=3,
    balls=60,
    target=150
)

p_win = monte_carlo_win_prob(initial, N=100000)
print("Win Probability:", round(p_win, 3))

NameError: name 'init_state' is not defined